# Prompt Engineering Part 2: Model Parameters

This notebook experiments with temperature, top_p, and penalties to see how model behavior changes under different settings.

## Objective
Through applied experimentation, we will:
1. Compare the effect of temperature on the same prompt.
2. Compare the effect of top_p while keeping temperature fixed.
3. Test how presence_penalty and frequency_penalty reduce repetition.
4. Answer the theory questions in my own words and reflect on the results.

## 0) Setup and Configuration

**Important**: Run this cell first. It loads your environment and sets up the reusable `call_model` helper function for parameter testing.

In [1]:
"""Setup cell for Part 2: model parameters.

What this cell does:
1) Load credentials and configuration from .env
2) Validate required variables
3) Create the OpenAI client for Azure OpenAI-compatible endpoint
4) Expose a reusable helper function that supports temperature, top_p, and penalties
5) Verify connectivity with a test call
"""

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

AZURE_API_KEY = os.getenv("AZURE_API_KEY", "").strip()
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT", "").strip()
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "").strip()

missing = [
    name
    for name, value in [
        ("AZURE_API_KEY", AZURE_API_KEY),
        ("AZURE_ENDPOINT", AZURE_ENDPOINT),
        ("DEPLOYMENT_NAME", DEPLOYMENT_NAME),
    ]
    if not value
]

if missing:
    raise ValueError(
        "Missing required environment variables: " + ", ".join(missing) +
        ". Fill them in your .env file and rerun this cell."
    )

print("Loaded environment variables successfully.")
print(f"Endpoint: {AZURE_ENDPOINT[:50]}...")
print(f"Model deployment: {DEPLOYMENT_NAME}")

client = OpenAI(
    base_url=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
)
print("Client initialized: OpenAI (Azure OpenAI-compatible endpoint)")


def call_model(
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.7,
    top_p: float = 1.0,
    presence_penalty: float = 0.0,
    frequency_penalty: float = 0.0,
    max_tokens: int = 500,
    print_prompts: bool = True,
) -> str:
    """Send a chat completion request and return assistant text content."""
    if print_prompts:
        print("\n" + "=" * 60)
        print("SYSTEM PROMPT:")
        print(system_prompt)
        print("\nUSER PROMPT:")
        print(user_prompt)
        print("=" * 60 + "\n")

    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        top_p=top_p,
        presence_penalty=presence_penalty,
        frequency_penalty=frequency_penalty,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


try:
    test_output = call_model(
        system_prompt="You are a concise assistant.",
        user_prompt="Reply with exactly: Connection OK",
        temperature=0.0,
        top_p=1.0,
        presence_penalty=0.0,
        frequency_penalty=0.0,
        max_tokens=20,
        print_prompts=False,
    )
    print("✓ Connectivity test passed.")
    print(f"Test response: {test_output}")
except Exception as exc:
    print("✗ Connectivity test failed.")
    print(f"Error type: {type(exc).__name__}")
    print(f"Error detail: {exc}")

Loaded environment variables successfully.
Endpoint: https://02-prompt-engineering-resource.openai.azur...
Model deployment: gpt-4o
Client initialized: OpenAI (Azure OpenAI-compatible endpoint)
✓ Connectivity test passed.
Test response: Connection OK


---

## 1) Temperature Experimentation

**The Idea:**
Temperature controls how random or creative the output is. I will keep the prompt the same and compare several temperature values to see how the tone and variety change.

**Prompt Focus:**
A short marketing-style line for a reusable water bottle brand.

In [3]:
"""Temperature experiment.

Same prompt, different temperature values.
"""

system_prompt_temp = """
You are a concise marketing copywriter for outdoor and lifestyle products.
"""

user_prompt_temp = """
Write one short marketing line for a reusable water bottle brand.
"""

temperatures = [0.0, 0.5, 1.0, 1.5]

for temperature_value in temperatures:
    response = call_model(
        system_prompt_temp,
        user_prompt_temp,
        temperature=temperature_value,
        top_p=1.0,
        presence_penalty=0.0,
        frequency_penalty=0.0,
        max_tokens=40,
        print_prompts=False,
    )
    print(f"TEMPERATURE = {temperature_value}")
    print(response)
    print("-" * 60)

TEMPERATURE = 0.0
"Stay hydrated, save the planet—sip sustainably with our reusable water bottles!"
------------------------------------------------------------
TEMPERATURE = 0.5
"Sip sustainably, hydrate endlessly—your perfect reusable companion."
------------------------------------------------------------
TEMPERATURE = 1.0
"Hydrate sustainably, adventure endlessly—refill with purpose."
------------------------------------------------------------
TEMPERATURE = 1.5
"Stay hydrated, save the planet—one sip at a time."
------------------------------------------------------------


#### Comparative Analysis

At low temperature, the output was more predictable and close to the same style each time. As temperature increased, the wording became more varied and creative.

For a simple marketing line, lower temperatures felt safer and more controlled. Higher temperatures gave more personality, but they also made the output less consistent.

I would use low temperature for factual or structured tasks and higher temperature for creative writing or brainstorming.

---

## 2) top_p Experimentation

**The Idea:**
I will keep temperature fixed and vary top_p to see how the model changes its selection of likely words while staying at the same randomness level.

**Prompt Focus:**
A short line about a productivity app for busy students.

In [4]:
"""top_p experiment.

Same prompt, temperature fixed at 1.0, top_p changes.
"""

system_prompt_top_p = """
You are a concise marketing copywriter for digital products.
"""

user_prompt_top_p = """
Write one short line about a productivity app for busy students.
Keep it under 15 words.
"""

top_p_values = [0.1, 0.5, 0.9, 1.0]

for top_p_value in top_p_values:
    response = call_model(
        system_prompt_top_p,
        user_prompt_top_p,
        temperature=1.0,
        top_p=top_p_value,
        presence_penalty=0.0,
        frequency_penalty=0.0,
        max_tokens=40,
        print_prompts=False,
    )
    print(f"TOP_P = {top_p_value}")
    print(response)
    print("-" * 60)

TOP_P = 0.1
"Stay organized and ace your studies with our all-in-one productivity app!"
------------------------------------------------------------
TOP_P = 0.5
"Master your schedule and ace deadlines with ease—productivity made simple for students!"
------------------------------------------------------------
TOP_P = 0.9
"Master your schedule and ace deadlines with ease—your ultimate student productivity app!"
------------------------------------------------------------
TOP_P = 1.0
"Stay organized and ace your studies with a productivity app built for busy students!"
------------------------------------------------------------


#### Comparative Analysis

With a low top_p value, the responses were narrower and more conservative. Higher values allowed the model to choose from a wider set of likely words, so the wording became more varied.

The main difference from temperature is that top_p changes the pool of allowed words instead of just turning randomness up or down. That made the outputs feel more controlled than temperature changes at the same level of creativity.

I would keep top_p near 1.0 unless I specifically want to limit the output to safer, more likely wording.

---

## 3) Penalties Experimentation

**The Idea:**
I will test presence_penalty and frequency_penalty on a prompt that tends to repeat similar ideas. This helps show which penalty reduces repetition more clearly.

**Prompt Focus:**
Generate short reasons to buy a reusable water bottle.

In [5]:
"""Penalty experiment.

Compare repetition with and without penalties.
"""

system_prompt_penalty = """
You are a product marketing assistant.
"""

user_prompt_penalty = """
Generate 8 short reasons to buy a reusable water bottle.
Make each reason distinct and avoid repeating the same wording.
"""

configs = [
    ("BASELINE", 0.0, 0.0),
    ("PRESENCE_PENALTY", 0.6, 0.0),
    ("FREQUENCY_PENALTY", 0.0, 0.8),
    ("BOTH_PENALTIES", 0.6, 0.8),
]

for label, presence_penalty_value, frequency_penalty_value in configs:
    response = call_model(
        system_prompt_penalty,
        user_prompt_penalty,
        temperature=0.9,
        top_p=1.0,
        presence_penalty=presence_penalty_value,
        frequency_penalty=frequency_penalty_value,
        max_tokens=200,
        print_prompts=False,
    )
    print(f"{label} | presence_penalty={presence_penalty_value} | frequency_penalty={frequency_penalty_value}")
    print(response)
    print("-" * 60)

BASELINE | presence_penalty=0.0 | frequency_penalty=0.0
1. **Eco-Friendly Choice**: Reduce plastic waste and help protect the planet by using a reusable water bottle.  
2. **Cost Savings**: Save money over time by refilling your bottle instead of buying disposable ones.  
3. **Durability**: Most reusable water bottles are made to last, resisting dents and damage for long-term use.  
4. **Customizable Designs**: Express your personality with a variety of colors, materials, and styles.  
5. **Better Hydration**: Having a reusable bottle on hand encourages you to drink more water throughout the day.  
6. **Improved Taste**: Many bottles come with filters or insulation to keep your water clean and flavorful.  
7. **Temperature Control**: Keep drinks hot or cold for hours with insulated reusable bottles.  
8. **Health Benefits**: Avoid harmful chemicals like BPA that can be found in single-use plastic bottles.  
------------------------------------------------------------
PRESENCE_PENALTY |

#### Comparative Analysis

Presence penalty encouraged the model to introduce new ideas, while frequency penalty helped reduce repeated wording in the list.

Using both together gave the most varied responses, but it also changed the style more noticeably than either penalty alone.

I would use presence penalty when I want novelty and frequency penalty when I want less repetition in longer outputs.

### Theory Questions

1. **What is the difference between top_p and temperature?**
   Temperature changes how random the model is overall. top_p changes which words are allowed by limiting the probability pool.

2. **Why is it not recommended to tune top_p and temperature at the same time?**
   Because both settings affect randomness, changing them together makes it harder to know which one caused the result.

3. **What is the difference between presence_penalty and frequency_penalty?**
   presence_penalty pushes the model to introduce new topics or words. frequency_penalty pushes it to repeat existing words less often.

## 5) Conclusions

For coding and technical tasks, I would keep temperature low and use top_p near 1.0 for stable answers.
For creativity, I would raise temperature and sometimes top_p to get more varied ideas.
For data extraction or structured output, I would keep both settings conservative and rely more on explicit prompting.
For repetitive generation, I would use presence_penalty and frequency_penalty to reduce repeated phrasing.